<a href="https://colab.research.google.com/github/annchristy-sys/misdiagnosis-health-equity-dashboard/blob/main/cleaning_up_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sklearn as sk
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
cvone=pd.read_csv("1_disease_error_harm.csv")

In [ ]:
cvone

,disease,category,incidence_k,error_rate_pct,errors_k,harm_rate_pct,serious_harms_k,focus_flag,source
0,Stroke,Vascular,952,17.5,166,9.8,94,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
1,Venous thromboembolism,Vascular,320,20.4,65,10.9,35,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
2,Arterial thromboembolism,Vascular,173,23.9,41,12.7,22,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
3,Aortic aneurysm & dissection,Vascular,96,35.6,34,22.1,21,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
4,Myocardial infarction,Vascular,1242,1.5,19,0.8,10,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
5,Sepsis,Infection,1345,9.9,134,5.9,79,Sepsis,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
6,Pneumonia,Infection,1469,9.5,140,4.6,68,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
7,Meningitis & encephalitis,Infection,47,25.6,12,14.7,7,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
8,Spinal abscess,Infection,14,62.1,8,36.7,5,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"
9,Endocarditis,Infection,34,25.5,9,13.8,5,NaN,"Newman-Toker et al., BMJ Qual Saf 2023 (Table 1)"


In [ ]:
cvtwo=pd.read_csv("2_national_scale.csv")
cvthree=pd.read_csv("3_demographic_disparities.csv")
cvfour=pd.read_csv("4_sepsis_context.csv")
cvfive=pd.read_csv("5_race_sex_disparities.csv")
cvsix=pd.read_csv("6_stage_at_diagnosis.csv")

In [ ]:
cvthree

,group,comparison,metric,value,unit,condition_context,source
0,Women,vs white men,Higher likelihood of misdiagnosis,20.0,percent (low end),All conditions,"Szabo, KFF Health News / NBC News, Jan 2024"
1,Women,vs white men,Higher likelihood of misdiagnosis,30.0,percent (high end),All conditions,"Szabo, KFF Health News / NBC News, Jan 2024"
2,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,20.0,percent (low end),All conditions,"Szabo, KFF Health News / NBC News, Jan 2024"
3,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,30.0,percent (high end),All conditions,"Szabo, KFF Health News / NBC News, Jan 2024"
4,Non-Hispanic Black mothers,vs non-Hispanic white mothers,Maternal mortality relative risk,2.6,x times,Maternal / peripartum cardiomyopathy,CDC NCHS 2021 (cited in KFF/NBC 2024)
5,Black patients,vs white patients,5-year melanoma mortality relative risk,3.0,x times,Melanoma,AAMC (cited in KFF/NBC 2024)
6,Dark-skin images in medical textbooks,share of all images,Representation,4.5,percent,Training materials,J Am Acad Dermatol 2020 (cited in KFF/NBC 2024)
7,Hospital patients who died/transferred to ICU,had a diagnostic error,Error prevalence,23.0,percent (~1 in 4),Inpatient,"Auerbach et al., JAMA Intern Med Jan 2024"


In [ ]:
cvfive

,condition,metric,group,comparison_group,value,unit,source_short,source_url
0,Sepsis,Sepsis-related mortality relative risk,Black Americans,White Americans,2.00,x times,Sepsis race meta-analysis (PubMed 31144133),https://pubmed.ncbi.nlm.nih.gov/31144133/
1,Sepsis,Sepsis incidence & mortality risk (range),Black Americans,White Americans,1.50,x times (low end),Racial disparities in sepsis (PMC6315577),https://pmc.ncbi.nlm.nih.gov/articles/PMC6315577/
2,Sepsis,Sepsis incidence & mortality risk (range),Black Americans,White Americans,3.50,x times (high end),Racial disparities in sepsis (PMC6315577),https://pmc.ncbi.nlm.nih.gov/articles/PMC6315577/
3,Breast cancer,Breast cancer death risk,Black women,White women,38.00,percent higher,American Cancer Society 2022-2024,https://www.cancer.org/research/acs-research-n...
4,Breast cancer,Breast cancer incidence,Black women,White women,-5.00,percent (lower),American Cancer Society 2022-2024,https://www.cancer.org/research/acs-research-n...
5,Breast cancer,"Death rate, women under 50",Black women,White women,2.00,x times,American Cancer Society 2022-2024,https://www.cancer.org/research/acs-research-n...
6,Breast cancer,Triple-negative / aggressive subtype rate,Black women,White women,2.00,x times,American Cancer Society 2022-2024,https://www.cancer.org/research/acs-research-n...
7,Cervical cancer,Advanced-stage diagnosis odds ratio,Black women,White women,1.18,odds ratio,Cervical cancer inequities (PMC10726717),https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1...


In [ ]:
cvthree_renamed = cvthree.rename(columns={
    "condition_context": "condition",
    "source": "source"          # already matches, kept for clarity
})
# cvthree has no source_url column — add one filled with blanks
if "source_url" not in cvthree_renamed.columns:
    cvthree_renamed["source_url"] = pd.NA

cvfive_renamed = cvfive.rename(columns={
    "comparison_group": "comparison",
    "source_short": "source"
})

In [ ]:
unified_cols = ["condition", "group", "comparison", "metric", "value", "unit", "source", "source_url"]

cvthree_final = cvthree_renamed[unified_cols]
cvfive_final = cvfive_renamed[unified_cols]

In [ ]:
combined = pd.concat([cvthree_final, cvfive_final], ignore_index=True)

In [ ]:
print("Combined shape:", combined.shape)
print("\nUnique metrics (check these don't need standardizing):")
print(combined["metric"].unique())
print("\nUnique units:")
print(combined["unit"].unique())
print("\nRows with missing source_url (expected for cvthree rows):")
print(combined[combined["source_url"].isna()].shape[0])

Combined shape: (16, 8)

Unique metrics (check these don't need standardizing):
['Higher likelihood of misdiagnosis' 'Maternal mortality relative risk'
 '5-year melanoma mortality relative risk' 'Representation'
 'Error prevalence' 'Sepsis-related mortality relative risk'
 'Sepsis incidence & mortality risk (range)' 'Breast cancer death risk'
 'Breast cancer incidence' 'Death rate, women under 50'
 'Triple-negative / aggressive subtype rate'
 'Advanced-stage diagnosis odds ratio']

Unique units:
['percent (low end)' 'percent (high end)' 'x times' 'percent'
 'percent (~1 in 4)' 'x times (low end)' 'x times (high end)'
 'percent higher' 'percent (lower)' 'odds ratio']

Rows with missing source_url (expected for cvthree rows):
8


In [ ]:
combined

,condition,group,comparison,metric,value,unit,source,source_url
0,All conditions,Women,vs white men,Higher likelihood of misdiagnosis,20.00,percent (low end),"Szabo, KFF Health News / NBC News, Jan 2024",NaN
1,All conditions,Women,vs white men,Higher likelihood of misdiagnosis,30.00,percent (high end),"Szabo, KFF Health News / NBC News, Jan 2024",NaN
2,All conditions,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,20.00,percent (low end),"Szabo, KFF Health News / NBC News, Jan 2024",NaN
3,All conditions,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,30.00,percent (high end),"Szabo, KFF Health News / NBC News, Jan 2024",NaN
4,Maternal / peripartum cardiomyopathy,Non-Hispanic Black mothers,vs non-Hispanic white mothers,Maternal mortality relative risk,2.60,x times,CDC NCHS 2021 (cited in KFF/NBC 2024),NaN
5,Melanoma,Black patients,vs white patients,5-year melanoma mortality relative risk,3.00,x times,AAMC (cited in KFF/NBC 2024),NaN
6,Training materials,Dark-skin images in medical textbooks,share of all images,Representation,4.50,percent,J Am Acad Dermatol 2020 (cited in KFF/NBC 2024),NaN
7,Inpatient,Hospital patients who died/transferred to ICU,had a diagnostic error,Error prevalence,23.00,percent (~1 in 4),"Auerbach et al., JAMA Intern Med Jan 2024",NaN
8,Sepsis,Black Americans,White Americans,Sepsis-related mortality relative risk,2.00,x times,Sepsis race meta-analysis (PubMed 31144133),https://pubmed.ncbi.nlm.nih.gov/31144133/
9,Sepsis,Black Americans,White Americans,Sepsis incidence & mortality risk (range),1.50,x times (low end),Racial disparities in sepsis (PMC6315577),https://pmc.ncbi.nlm.nih.gov/articles/PMC6315577/


In [ ]:
newcombined = combined.drop(columns=["source", "source_url"])
newcombined = newcombined[~newcombined["condition"].isin(["Training materials", "Inpatient"])]
newcombined = newcombined.reset_index(drop=True)

print(newcombined.shape)
newcombined


(14, 6)


,condition,group,comparison,metric,value,unit
0,All conditions,Women,vs white men,Higher likelihood of misdiagnosis,20.00,percent (low end)
1,All conditions,Women,vs white men,Higher likelihood of misdiagnosis,30.00,percent (high end)
2,All conditions,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,20.00,percent (low end)
3,All conditions,Racial & ethnic minorities,vs white men,Higher likelihood of misdiagnosis,30.00,percent (high end)
4,Maternal / peripartum cardiomyopathy,Non-Hispanic Black mothers,vs non-Hispanic white mothers,Maternal mortality relative risk,2.60,x times
5,Melanoma,Black patients,vs white patients,5-year melanoma mortality relative risk,3.00,x times
6,Sepsis,Black Americans,White Americans,Sepsis-related mortality relative risk,2.00,x times
7,Sepsis,Black Americans,White Americans,Sepsis incidence & mortality risk (range),1.50,x times (low end)
8,Sepsis,Black Americans,White Americans,Sepsis incidence & mortality risk (range),3.50,x times (high end)
9,Breast cancer,Black women,White women,Breast cancer death risk,38.00,percent higher


In [ ]:
data = [
    {"condition": "All conditions", "group": "Women", "comparison": "Men (White)", "times_as_likely": 1.25,
     "metric": "Likely to be misdiagnosed"},

    {"condition": "All conditions", "group": "Racial & ethnic minorities", "comparison": "Men (White)", "times_as_likely": 1.25,
     "metric": "Likely to be misdiagnosed"},

    {"condition": "Pregnancy-related heart failure (peripartum cardiomyopathy)",
     "group": "Black mothers", "comparison": "White mothers","times_as_likely": 2.6,
     "metric": "Likely to die from this condition"},

    {"condition": "Melanoma (skin cancer)", "group": "Black patients", "comparison": "White patients", "times_as_likely": 3.0,
     "metric": "Likely to die within 5 years of diagnosis"},

    {"condition": "Sepsis (severe infection)", "group": "Black Americans", "comparison": "White Americans", "times_as_likely": 2.0,
     "metric": "Likely to die from sepsis"},

    {"condition": "Sepsis (severe infection)", "group": "Black Americans", "comparison": "White Americans", "times_as_likely": 2.5,
     "metric": "Likely to get or die from sepsis (broader range across studies)"},

    {"condition": "Breast cancer", "group": "Black women", "comparison": "White women", "times_as_likely": 1.38,
     "metric": "Likely to die from breast cancer"},

    {"condition": "Breast cancer", "group": "Black women", "comparison": "White women", "times_as_likely": 0.95,
     "metric": "Likely to be diagnosed with breast cancer"},

    {"condition": "Breast cancer", "group": "Black women", "comparison": "White women", "times_as_likely": 2.0,
     "metric": "Likely to die if diagnosed before age 50"},

    {"condition": "Breast cancer", "group": "Black women", "comparison": "White women", "times_as_likely": 2.0,
     "metric": "Likely to have an aggressive (triple-negative) tumor"},

    {"condition": "Cervical cancer", "group": "Black women", "comparison": "White women","times_as_likely": 1.18,
     "metric": "Likely to be diagnosed at an advanced stage"}
]

cleaned = pd.DataFrame(data)
cleaned


,condition,group,comparison,times_as_likely,metric
0,All conditions,Women,Men (White),1.25,Likely to be misdiagnosed
1,All conditions,Racial & ethnic minorities,Men (White),1.25,Likely to be misdiagnosed
2,Pregnancy-related heart failure (peripartum ca...,Black mothers,White mothers,2.60,Likely to die from this condition
3,Melanoma (skin cancer),Black patients,White patients,3.00,Likely to die within 5 years of diagnosis
4,Sepsis (severe infection),Black Americans,White Americans,2.00,Likely to die from sepsis
5,Sepsis (severe infection),Black Americans,White Americans,2.50,Likely to get or die from sepsis (broader rang...
6,Breast cancer,Black women,White women,1.38,Likely to die from breast cancer
7,Breast cancer,Black women,White women,0.95,Likely to be diagnosed with breast cancer
8,Breast cancer,Black women,White women,2.00,Likely to die if diagnosed before age 50
9,Breast cancer,Black women,White women,2.00,Likely to have an aggressive (triple-negative)...


In [ ]:
othercleaned = cvone.drop(columns=["source", "focus_flag"])
othercleaned

,disease,category,incidence_k,error_rate_pct,errors_k,harm_rate_pct,serious_harms_k
0,Stroke,Vascular,952,17.5,166,9.8,94
1,Venous thromboembolism,Vascular,320,20.4,65,10.9,35
2,Arterial thromboembolism,Vascular,173,23.9,41,12.7,22
3,Aortic aneurysm & dissection,Vascular,96,35.6,34,22.1,21
4,Myocardial infarction,Vascular,1242,1.5,19,0.8,10
5,Sepsis,Infection,1345,9.9,134,5.9,79
6,Pneumonia,Infection,1469,9.5,140,4.6,68
7,Meningitis & encephalitis,Infection,47,25.6,12,14.7,7
8,Spinal abscess,Infection,14,62.1,8,36.7,5
9,Endocarditis,Infection,34,25.5,9,13.8,5


In [ ]:
othercleaned.to_csv("harm_errors_diagnosing.csv", index=False)
print("Saved harm_errors_diagnosing.csv")

Saved harm_errors_diagnosing.csv


In [ ]:
cleaned.to_csv("disparity_data_plain_reading.csv", index=False)
print("Saved disparity_data_plain_reading.csv")


Saved disparity_data_plain_reading.csv


Most of the detailed disparity statistics in this dashboard come from studies published between 2012 and 2024 — this is the most rigorous data available, since studies of this scale take years to collect and verify. Newer evidence confirms the underlying problem hasn't gone away: as of 2026, hospitals are still running active initiatives to close these same gaps in sepsis care (Linnander, Yale School of Public Health, Apr 2026), and Susan G. Komen's February 2026 report confirms the breast cancer mortality gap remains unresolved, prompting new 2026 policy pushes to remove screening barriers. In short, the numbers are older, but the issue they describe is current, and it spans multiple disease categories, not just one.

Something worth noting for improving is in the start of medical school in general, representation is lacking in medical textbooks (research more into this: https://news.mit.edu/2024/doctors-more-difficulty-diagnosing-diseases-images-darker-skin-0205)